In [1]:
import os 
os.chdir('../../../')
os.environ["DPM_TQDM"] = "False"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

!nvidia-smi

Wed Aug 20 07:05:17 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:02:00.0 Off |                  Off |
| 31%   40C    P8             12W /  450W |      25MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# -*- coding: utf-8 -*-
import os
import math
import numpy as np
from easydict import EasyDict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

In [3]:
!ls /dataset/sana/

valid4.5  valid4.5.zip


In [4]:
# ===============================
# Config
# ===============================
config = EasyDict()
config.backbone      = 'SANA'
config.train_pt_dir  = '/dataset/sana/train4.5_4090'
config.valid_pt_dir  = '/dataset/sana/valid4.5'
config.batch_size    = 1
config.CFG           = 4.5
config.val_every     = 500
config.log_dir       = "logs/sana/0820-1:SANA,CLIP,6steps,BLIP"
config.latent_size = (32, 16, 16)

# for Solver
config.solver = EasyDict()
config.solver.steps = 5
config.solver.skip_type = 'time_uniform_flow'
config.solver.flow_shift = 3.0
config.solver.pred_order = 1
config.solver.corr_order = 2

# LR & Scheduler
config.base_lr       = 2e-3
config.total_steps   = 100*1000        # 전체 학습 스텝

# Loss
config.classifier = EasyDict()

config.losses = ['inception', 'PSNR', 'blip']
config.main_loss = 'blip'

os.makedirs(config.log_dir, exist_ok=True)


In [5]:
# ===============================
# Model (frozen)
# ===============================
from backbones.dit import DiT
from backbones.sana import SANA
from utils.inception import FIDInception
from utils.clip import CLIPEmbedder
from utils.blip import BLIPTextLikelihood

if config.backbone == 'DiT':
    model = DiT(trainable=True)
elif config.backbone == 'SANA':
    model = SANA(trainable=True)
model.set_freeze()
device = model.device
print(model)

inception = FIDInception().to(device)
if 'clip' in config.losses:
    clip = CLIPEmbedder(device=model.device)
if 'blip' in config.losses:
    blip = BLIPTextLikelihood(device=model.device)
print('done')

/home/scpark/miniconda3/envs/sana/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading pipeline components...: 100%|██████████| 5/5 [00:00<00:00,  5.75it/s]


done


In [6]:
# ===============================
# Solver / Optimizer / Scheduler
# ===============================
from solvers.taylor.solver.gdual_solver import GDual_Solver
from solvers.taylor.transform.logaffine_transform import LogAffineTransform
from solvers.taylor.extractor.table_extractor import Extractor

noise_schedule = model.get_noise_schedule()
extractor = Extractor()
transform = LogAffineTransform(gamma_push=True, gamma_max=2, tau_offset=1, kappa_max=2, eps=1e-2)
solver = GDual_Solver(
    noise_schedule,
    steps=config.solver.steps,
    transform=transform,
    param_extractor=extractor,
    skip_type=config.solver.skip_type,
    flow_shift=config.solver.flow_shift,
    pred_order=config.solver.pred_order,
    corr_order=config.solver.corr_order,
    order1_kappa=True,
    order2_kappa=True,
    use_corrector=True,
    time_learning=True,
    train_mode=True
).to(device)

optimizer = torch.optim.AdamW(solver.parameters(), lr=config.base_lr)
print('solver/optimizer')

solver/optimizer


In [8]:
# ===============================
# Dataset / Dataloader
# ===============================
from datasets.pt_dataset import PtDataset

if config.main_loss not in ['clip', 'blip']:
    train_dataset = PtDataset(config.train_pt_dir)
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=8,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4,
    )
    print('len(train_dataset) :', len(train_dataset))
    
valid_dataset = PtDataset(config.valid_pt_dir, n_files=100)
print('len(valid_dataset) :', len(valid_dataset))

valid_loader = DataLoader(valid_dataset, batch_size=config.batch_size, shuffle=False)
print('dataloaders ready')


len(valid_dataset) : 100
dataloaders ready


In [9]:
# ===============================
# Utils
# ===============================

def abort_if_bad(tag, value, step=None):
    v = float(value.detach().cpu()) if isinstance(value, torch.Tensor) else float(value)
    if (not math.isfinite(v)) or (v >= 100.0):
        msg = f"[EARLY-STOP] {tag} loss={v:.6f}" + (f" @ step {step}" if step is not None else "")
        print(msg, flush=True)
        raise RuntimeError(msg)

def save_checkpoint(global_step, save_dir, solver, optimizer):
    ckpt = {
        "global_step": int(global_step),
        "optim_state_dict": optimizer.state_dict(),
        "solver_state_dict": solver.state_dict(),
        "config": dict(config),
    }
    os.makedirs(save_dir, exist_ok=True)
    step_path = os.path.join(save_dir, f"step_{global_step:08d}.pt")
    torch.save(ckpt, step_path)
    return step_path

In [10]:
from IPython.display import clear_output

@torch.no_grad()
def get_valid_loss(device, solver):
    solver.eval()
    losses = {}
    if 'PSNR' in config.losses:
        losses['PSNR'] = []
    if 'inception' in config.losses:
        losses['inception'] = []
    if 'clip' in config.losses:
        losses['clip'] = []
    if 'blip' in config.losses:
        losses['blip'] = []
    
    for i, batch in enumerate(valid_loader):
        noises = batch['noise'].to(device, non_blocking=True)
        conds  = batch['cond']
        targets= batch['sample'].to(device, non_blocking=True)
        target_features= batch['inception_feature'].to(device, non_blocking=True)
        
        model_fn = model.get_model_fn(noise_schedule, pos_conds=conds, guidance_scale=config.CFG)
        with torch.no_grad():
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                latent_pred = solver.sample(noises, model_fn)
                if 'PSNR' in losses:
                    psnr_loss = torch.log(F.mse_loss(latent_pred, targets) + 1e-8)
                    losses['PSNR'].append(psnr_loss.item())

                if 'inception' in config.losses or 'clip' in config.losses or 'blip' in config.losses:
                    sample_pred = model.decode_vae(latent_pred, raw_output=True)
        
                    if 'inception' in config.losses:
                        pred = inception(sample_pred)
                        inception_loss = F.mse_loss(pred, target_features)
                        losses['inception'].append(inception_loss.item())

                    if 'clip' in config.losses:
                        clip_loss = clip.get_cossim_loss(sample_pred, conds)
                        losses['clip'].append(clip_loss.item())
                        print('valid :', i, clip_loss)

                    if 'blip' in config.losses:
                        blip_loss = blip.text_nll(sample_pred, conds)
                        losses['blip'].append(blip_loss.item())
                        print('valid :', i, blip_loss)
    clear_output()

    for key in losses:
        losses[key] = float(np.mean(losses[key]))
    return losses

In [11]:
def do_train_loop(device, writer, solver, optimizer, global_step):
    solver.train()
    if config.main_loss in ['clip', 'blip']:
        prompts = np.load('prompts/mscoco2017.npz')['arr_0'].tolist()
        pbar = tqdm(range(100))
    else:
        pbar = tqdm(train_loader)
        
    for _, batch in enumerate(pbar):
        if global_step >= config.total_steps:
            break

        #if global_step > 0 and global_step % config.val_every == 0:
        if global_step % config.val_every == 0:
            valid_losses = get_valid_loss(device, solver)
            for key in valid_losses:
                writer.add_scalar(key, valid_losses[key], global_step)
            save_checkpoint(global_step, config.log_dir, solver, optimizer)

        optimizer.zero_grad(set_to_none=True)
        if config.main_loss in ['clip', 'blip']:
            noises = torch.randn(config.batch_size, *config.latent_size).to(device, non_blocking=True)
            indexes = np.random.randint(0, len(prompts), size=(len(noises),))
            conds = [prompts[index] for index in indexes]
        else:
            noises = batch['noise'].to(device, non_blocking=True)
            conds  = batch['cond']
            targets= batch['sample'].to(device, non_blocking=True)
            target_features = batch['inception_feature'][:, 0].to(device, non_blocking=True)

        model_fn = model.get_model_fn(noise_schedule, pos_conds=conds, guidance_scale=config.CFG)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            latent_pred = solver.sample(noises, model_fn)
            if 'PSNR' == config.main_loss:
                loss = torch.log(F.mse_loss(latent_pred, targets) + 1e-8)
                
            if config.main_loss in ['inception', 'clip', 'blip']:
                sample_pred = model.decode_vae(latent_pred, raw_output=True)
    
                if 'inception' == config.main_loss:
                    pred = inception(sample_pred)
                    loss = F.mse_loss(pred, target_features)
                    
                if 'clip' == config.main_loss:
                    loss = clip.get_cossim_loss(sample_pred, conds)

                if 'blip' == config.main_loss:
                    loss = blip.text_nll(sample_pred, conds)
                    
        abort_if_bad("train", loss, global_step)  # ← 즉시 중단

        loss.backward()
        
        # ---- 2) grad norm 기준 클리핑
        torch.nn.utils.clip_grad_norm_(solver.parameters(), 1.0)
        optimizer.step()
        
        lr_now = optimizer.param_groups[0]["lr"]
        pbar.set_postfix({'loss': loss.item(), 'lr': lr_now})
        global_step += 1
        clear_output()

    return global_step



In [12]:
# ===============================
# Train
# ===============================
def main():
    writer = SummaryWriter(config.log_dir)
    print('tensorboard:', config.log_dir)

    global_step = 0
    while True:
        if global_step >= config.total_steps:
            break
        global_step = do_train_loop(device, writer, solver, optimizer, global_step)
    print('E-N-D')
    
if __name__ == "__main__":
    main()


tensorboard: logs/sana/0820-1:SANA,CLIP,6steps,BLIP


  0%|          | 0/100 [00:00<?, ?it/s]Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


valid : 0 tensor(4.0646, device='cuda:0')
valid : 1 tensor(2.1730, device='cuda:0')
valid : 2 tensor(3.3449, device='cuda:0')
valid : 3 tensor(3.4105, device='cuda:0')
valid : 4 tensor(0.8638, device='cuda:0')
valid : 5 tensor(2.1653, device='cuda:0')
valid : 6 tensor(2.1662, device='cuda:0')
valid : 7 tensor(4.2207, device='cuda:0')
valid : 8 tensor(3.3615, device='cuda:0')
valid : 9 tensor(3.5768, device='cuda:0')
valid : 10 tensor(3.8645, device='cuda:0')
valid : 11 tensor(3.4480, device='cuda:0')
valid : 12 tensor(3.5899, device='cuda:0')
valid : 13 tensor(3.2760, device='cuda:0')
valid : 14 tensor(2.0263, device='cuda:0')
valid : 15 tensor(2.6525, device='cuda:0')
valid : 16 tensor(3.2676, device='cuda:0')
valid : 17 tensor(2.1298, device='cuda:0')
valid : 18 tensor(1.9810, device='cuda:0')
valid : 19 tensor(1.4361, device='cuda:0')
valid : 20 tensor(2.9066, device='cuda:0')
valid : 21 tensor(3.2972, device='cuda:0')
valid : 22 tensor(3.1520, device='cuda:0')
valid : 23 tensor(0.8

  0%|          | 0/100 [00:08<?, ?it/s]

valid : 50 tensor(3.4273, device='cuda:0')


KeyboardInterrupt: 